# Face Recognition using PCA and KNN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from time import time

In [ ]:
def show_orignal_images(pixels):
    fig, axes = plt.subplots(3, 8, figsize=(9, 4),
                             subplot_kw={'xticks': [], 'yticks': []})
    for i, ax in enumerate(axes.flat):
        ax.imshow(np.array(pixels)[i].reshape(64, 64), cmap='gray')
    plt.show()

def show_eigenfaces(pca):
    fig, axes = plt.subplots(3, 8, figsize=(9, 4),
                             subplot_kw={'xticks': [], 'yticks': []})
    for i, ax in enumerate(axes.flat):
        ax.imshow(pca.components_[i].reshape(64, 64), cmap='gray')
        ax.set_title('PC ' + str(i + 1))
    plt.show()

In [ ]:
df = pd.read_csv('face_data.csv')
target = df['target'].values
pixels = df.drop(['target'], axis=1).values

print('Shape:', pixels.shape)
show_orignal_images(pixels)

x_train, x_test, y_train, y_test = train_test_split(
    pixels, target, test_size=0.25, random_state=42
)

n_components = 150
pca = PCA(n_components=n_components).fit(x_train)

plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.show()

show_eigenfaces(pca)

Xtrain_pca = pca.transform(x_train)
Xtest_pca = pca.transform(x_test)

print('Current shape of input data matrix:', Xtrain_pca.shape)

In [ ]:
plt.style.use('ggplot')

neighbors = np.arange(1, 9)
train_accuracy = np.empty(len(neighbors))
test_accuracy = np.empty(len(neighbors))

for i, k in enumerate(neighbors):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(x_train, y_train)
    train_accuracy[i] = knn.score(x_train, y_train)
    test_accuracy[i] = knn.score(x_test, y_test)

plt.title('k-NN Varying Number of Neighbors')
plt.plot(neighbors, test_accuracy, label='Testing Accuracy')
plt.plot(neighbors, train_accuracy, label='Training Accuracy')
plt.legend()
plt.xlabel('Number of Neighbors')
plt.ylabel('Accuracy')
plt.show()

knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(x_train, y_train)

print('Accuracy:', knn.score(x_test, y_test))

t0 = time()
y_pred = knn.predict(x_test)
print('done in %0.3fs' % (time() - t0))
print(classification_report(y_test, y_pred))